In [18]:
import pandas as pd
import re
from konlpy.tag import Okt
from collections import defaultdict
# 데이터 로드
df = pd.read_csv('daum_movie_review.csv')
okt = Okt()
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [20]:
# 명사기반 검색엔진 구현(형태소 분석기 응용) - 형태소 분석기를 통해서 조사가 무엇이든간에 핵심의미(명사)

# 역색인 inverted index 
iverted_index = defaultdict(list)
for idx, row in df.head(1000).iterrows():
    review = row['review']
    nouns = okt.nouns(review)
    for noun in set(nouns):
        if len(noun) > 1 :
            iverted_index[noun].append(idx)

def search_movie_review(query):
    '''질의어에서 명사 추출'''
    query_nouns = okt.nouns(query)
    # 첫번째 명사기준으로 검색(단순화)
    target_noun = query_nouns[0]
    if target_noun in iverted_index:
        match_index = iverted_index[target_noun]
        return df.loc[match_index][['review','rating']].head()
    else:
        return '검색 결과가 없습니다.'

search_movie_review('영화가')

,review,rating
21,롱턱 타노스의 장갑이 참 맘에 듬. 아이언 맨과 토르 닥터만 생고생하고.. 가...,6
26,이 영화를 보고나서 예전 영화 왓치맨이 생각나더군요 평화를 위해선 불특정 다수의 희...,9
34,어벤져스 답게 쏟아붓긴 했는데 한방이 없었다마블 영화들은 보고나면 늘 긴 예고편 본...,5
41,이건 뭐~ 이 정도 돈과 출연진 가지고 이렇게 망칠 수도 있구나를 보여주는 대표작...,2
44,마블 팬이라면 반드시 봐야하는 영화,10


In [26]:
df['review'][1]

'몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.'

In [27]:
# TTR 응용  고유토큰수 / 전체토큰수
def calc_ttr(text):
    tokens = okt.morphs(text)
    if not tokens: return 0
    return len(set(tokens)) /  len(tokens)

spam_review = '추천합니다 추천합니다 추천합니다 추천합니다 추천합니다'
normal_review = '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.'
calc_ttr(spam_review), calc_ttr(normal_review)

(0.2, 0.8571428571428571)

In [36]:
# review데이터 중에서 TTR이 0.4이하인 리뷰들만 추출
import re
def extract_hangule_blank(text):
    return re.sub(r'[^가-힣\s]','',text)

# df['clean_review'] = df['review'].apply(lambda x : extract_hangule_blank(x))
df['clean_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x))

In [39]:
df['ttr'] = df['clean_review'].apply(lambda x : calc_ttr(x) )

In [48]:
df[df['ttr'] <= 0.6]

,review,rating,date,title,clean_review,ttr
89,good,9,2018.08.04,인피니티 워,,0.0
103,ㅍㅎㅎㅎ,1,2018.08.02,인피니티 워,,0.0
147,................................!!!!!!!!!!!!!!...,0,2018.06.17,인피니티 워,,0.0
162,dksqhkwjdy,1,2018.06.06,인피니티 워,,0.0
167,ㅋㅋㅋ,9,2018.06.05,인피니티 워,,0.0
...,...,...,...,...,...,...
14051,remember me,9,2018.03.04,코코,,0.0
14248,...,9,2018.02.04,코코,,0.0
14317,ㅜ ㅜ,10,2018.01.29,코코,,0.0
14499,Good,10,2018.01.19,코코,,0.0


In [49]:
# 개인정보 마스킹
# 문의사항은 서울시 강남구 개포동 010-1234-1234로 연락주세요
import re
mobile_pattern = re.compile(r'(\b01[016789][-\s]?)(\d{3,4})([-\s]?\d{4}\b)')

def mask_mobile(text):
    return re.sub(mobile_pattern, r'\1****\3', text)
mask_mobile("연락처는 010-1234-5678 입니다")

'연락처는 010-****-5678 입니다'

In [50]:
df['review'].apply(lambda x : mask_mobile(x))

0                                   돈 들인건 티가 나지만 보는 내내 하품만
1             몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.
2        이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...
3                                      이 정도면 볼만하다고 할 수 있음!
4                                                     재미있다
                               ...                        
14720    어른들을 위한 동화    정말 오랜만에  좋은 애니를 보았습니다     가족의 소중...
14721                                     디즈니는 못해도 본전은 한다.
14722                              가족을 위한 영화... 괜찮은 영화.~~~
14723        간만에 제대로 잘짜여진 각본의 영화를 봤네 여운이 아직도 남아~어른을 위한 애니~
14724                     한국개봉을 눈빠지게 기다린 보람이있다 깨우치는게 많은 영화
Name: review, Length: 14725, dtype: object

In [ ]:
# 평정예측모델
# 1. 데이터 로드
# 2. 전처리
# 3. 벡터화
# 4. 차원축소(옵션)
# 5. 모델선택
# 6. 학습
# 7. 평가
# 8. 배포- 서비스(AWS)